# Generative AI 021 — Ollama and Open-Source LLMs

The first notebook in this track with a **real model**. Everything runs on your
own machine, with no API key.

| Part | What we check |
|---|---|
| A | a first call, and how many tokens per second your machine manages |
| B | temperature 0 gives the same answer every time |
| C | the Python library and a raw HTTP POST reach the same server |
| D | `generate` forgets; `chat` only remembers because you send the history |
| E | tool calling — and whether the model **invents** the price |
| F | what one SYSTEM line does to the output format |

**Before you start:** install Ollama from ollama.com, run `ollama pull llama3.2`
(about 2 GB), and `pip install ollama requests`. Figures in the lesson came from
a laptop with no GPU; yours will differ.

In [ ]:
import requests
try:
    tags = requests.get("http://localhost:11434/api/tags", timeout=3).json()
except requests.exceptions.ConnectionError:
    raise SystemExit("Ollama is not running. Start the Ollama app (or `ollama serve`) and re-run.")
names = [m["name"] for m in tags["models"]]
if not any(n.startswith("llama3.2") for n in names):
    raise SystemExit(f"llama3.2 is not pulled. Run `ollama pull llama3.2`. You have: {names}")
print("server up; models:", names)

## Part A — A first call

The first call loads the model into memory, so it is the slowest.

In [ ]:
import ollama

MODEL = "llama3.2"      # must already be pulled: `ollama pull llama3.2`

r = ollama.generate(model=MODEL, prompt="In two sentences, explain what photosynthesis is.",
                    options={"temperature": 0, "num_predict": 40})
print(r.response)

# The response also tells you how the model ran:
print(f"load {r.load_duration/1e9:.1f} s, {r.eval_count} tokens at "
      f"{r.eval_count/(r.eval_duration/1e9):.1f} tokens/sec")
# Measured on the course machine (CPU only): about 11 tokens/sec for this
# 3.2B model, and 2 tokens/sec for the 14.7B phi4. Yours will differ.

# stream=True yields chunks as they are produced:
for chunk in ollama.generate(model=MODEL, prompt="Count to five.",
                             stream=True, options={"num_predict": 20}):
    print(chunk.response, end="", flush=True)
print()

In [ ]:
assert r.eval_count > 0 and r.response.strip()
print(f"your machine: {r.eval_count/(r.eval_duration/1e9):.1f} tokens/sec "
      "(the course laptop, CPU only: 10.9)")

## Part B — Temperature 0 repeats itself

In [ ]:
prompt = "Write one short sentence about the monsoon."
for temp in (0.0, 1.5):
    outs = {ollama.generate(model=MODEL, prompt=prompt,
                            options={"temperature": temp, "num_predict": 30}).response.strip()
            for _ in range(5)}
    print(f"temperature {temp}: {len(outs)} distinct answers in 5 runs")
# temperature 0.0: 1 distinct answers in 5 runs
# temperature 1.5: 5 distinct answers in 5 runs
#
# Lesson 006's claim - temperature 0 gives the same output every time - seen
# with a real model rather than a softmax calculation.

In [ ]:
zero = {ollama.generate(model=MODEL, prompt=prompt,
                        options={"temperature": 0, "num_predict": 30}).response for _ in range(3)}
assert len(zero) == 1, "temperature 0 should give one answer"
print("temperature 0: one answer, every time")

## Part C — The same server over HTTP

In [ ]:
import requests

opts = {"temperature": 0, "seed": 7, "num_predict": 25}
q = "Explain black holes in one sentence."

via_library = ollama.generate(model=MODEL, prompt=q, options=opts).response
via_rest = requests.post("http://localhost:11434/api/generate",
                         json={"model": MODEL, "prompt": q, "options": opts,
                               "stream": False}).json()["response"]
print(via_library == via_rest)            # True

print([m["name"] for m in requests.get("http://localhost:11434/api/tags").json()["models"]])
# The library, the CLI and the app are all wrappers around this local server.

In [ ]:
assert via_library == via_rest
print("same server, same answer")

## Part D — generate against chat

In [ ]:
o = {"temperature": 0, "num_predict": 30}

# generate: every call stands alone
ollama.generate(model=MODEL, prompt="My name is Ayesha.", options=o)
print(ollama.generate(model=MODEL, prompt="What is my name?", options=o).response)
# "I don't have any information about your name..."

# chat: send the history
messages = [{"role": "user", "content": "My name is Ayesha."}]
reply = ollama.chat(model=MODEL, messages=messages, options=o).message.content
messages += [{"role": "assistant", "content": reply},
             {"role": "user", "content": "What is my name?"}]
print(ollama.chat(model=MODEL, messages=messages, options=o).message.content)
# "Your name is Ayesha."

In [ ]:
last = ollama.chat(model=MODEL, messages=messages, options=o).message.content
print("chat knew the name:", "Ayesha" in last)

## Part E — Tool calling with a real model

Read the second answer carefully. Did the model check the laptop's price, or
did it make one up? On the course machine it made one up.

In [ ]:
import json

INVENTORY = {"laptop": {"price": 55000, "stock": 3},
             "headphones": {"price": 2500, "stock": 10}}

# Step 1 - the tools: ordinary Python functions
def check_inventory(product: str) -> str:
    item = INVENTORY.get(product.lower().strip())
    return json.dumps(item | {"in_stock": item["stock"] > 0} if item
                      else {"product": product, "in_stock": False})

def calculate_loyalty_discount(base_price: int, years_as_customer: int) -> str:
    pct = min(years_as_customer * 2, 10)
    return json.dumps({"discount_percent": pct,
                       "final_price": base_price * (100 - pct) // 100})

FUNCS = {"check_inventory": check_inventory,
         "calculate_loyalty_discount": calculate_loyalty_discount}

# Step 2 - the schema: what the model actually reads
TOOLS = [
    {"type": "function", "function": {
        "name": "check_inventory", "description": "Check whether a product is in stock and its price",
        "parameters": {"type": "object", "properties": {"product": {"type": "string"}},
                       "required": ["product"]}}},
    {"type": "function", "function": {
        "name": "calculate_loyalty_discount",
        "description": "Calculate the final price after a loyalty discount",
        "parameters": {"type": "object",
                       "properties": {"base_price": {"type": "integer"},
                                      "years_as_customer": {"type": "integer"}},
                       "required": ["base_price", "years_as_customer"]}}},
]

def ask(question):
    messages = [{"role": "user", "content": question}]
    for _ in range(4):
        r = ollama.chat(model=MODEL, messages=messages, tools=TOOLS,     # Step 3
                        options={"temperature": 0})
        messages.append(r.message)
        if not r.message.tool_calls:
            return r.message.content
        for c in r.message.tool_calls:                                  # Step 4
            result = FUNCS[c.function.name](**c.function.arguments)
            print(f"  model asked for {c.function.name}({dict(c.function.arguments)}) -> {result}")
            messages.append({"role": "tool", "content": result})
    return "(stopped)"                                                  # Step 5: loop

print(ask("I want to buy an iPhone. Can you check stock?"))
print(ask("I have been a customer for 5 years. What would the laptop cost me?"))

# What llama3.2 did on the course machine for the second question:
#   model asked for calculate_loyalty_discount({'base_price': 1000, 'years_as_customer': 5})
#   "...a 10% discount on the base price of $1000, bringing the final price down to $900."
#
# It never called check_inventory. It INVENTED base_price=1000. The laptop
# costs 55,000, so the right answer is 49,500. The tool ran perfectly on a made-up
# input and the answer was fluent and wrong - lesson 019's problem, live.

In [ ]:
# Re-run the second question and look only at the arguments the model chose.
msgs = [{"role": "user", "content": "I have been a customer for 5 years. What would the laptop cost me?"}]
r = ollama.chat(model=MODEL, messages=msgs, tools=TOOLS, options={"temperature": 0})
calls = [(c.function.name, dict(c.function.arguments)) for c in (r.message.tool_calls or [])]
print("first tool calls:", calls)
invented = [a.get("base_price") for n, a in calls
            if n == "calculate_loyalty_discount" and a.get("base_price") != INVENTORY["laptop"]["price"]]
if invented:
    print(f"INVENTED: base_price={invented[0]}. The laptop costs 55,000; the right answer is 49,500.")
elif calls and calls[0][0] == "check_inventory":
    print("This model version checked the stock first. Good - but test it on more questions.")

## Part F — One SYSTEM line

In [ ]:
# Every model already has a Modelfile - read-only:
mf = ollama.show(MODEL).modelfile
DIRECTIVES = {"FROM", "PARAMETER", "TEMPLATE", "SYSTEM", "ADAPTER", "LICENSE", "MESSAGE"}
print(sorted({ln.split()[0] for ln in mf.splitlines()
              if ln and not ln[0].isspace() and ln.split()[0] in DIRECTIVES}))
# ['FROM', 'LICENSE', 'PARAMETER', 'TEMPLATE']

SYSTEM = ('You are a sentiment analysis API. You ONLY output JSON in this schema: '
          '{"sentiment": "positive|negative|neutral", "score": <number 0..1>}. No other text.')
reviews = ["I love this course, it is brilliant.", "The delivery was late and the box was broken.",
           "It arrived on Tuesday.", "Worst purchase I have made this year.",
           "Decent value for the price.", "The staff were so kind and helpful!"]

def valid_json(text):
    try:
        return json.loads(text.strip().strip("`").removeprefix("json").strip())["sentiment"] \
            in ("positive", "negative", "neutral")
    except Exception:
        return False

for label, system in (("no system", None), ("with SYSTEM", SYSTEM)):
    kw = {"system": system} if system else {}
    ok = sum(valid_json(ollama.generate(model=MODEL, prompt=f"Review: {rv}",
                                        options={"temperature": 0, "num_predict": 60},
                                        **kw).response) for rv in reviews)
    print(f"{label:<12} {ok}/6 valid JSON")
# no system    0/6 valid JSON
# with SYSTEM  6/6 valid JSON

# To bake that in as a named model, write a file called Modelfile:
#     FROM llama3.2
#     PARAMETER temperature 0.3
#     SYSTEM """...the same instruction..."""
# and run:  ollama create sentiment -f Modelfile
# The weights are untouched; only how they are used changes.

In [ ]:
print(f"with SYSTEM: {ok}/6 valid JSON (course laptop: 6/6; without it: 0/6)")

## What to take away

- Ollama runs open-weight models on **your** machine behind one local server,
  `localhost:11434`. The CLI, the library and LangChain all talk to it.
- A 4-bit quantisation stores about 5 bits per weight — the reason a 14.7B
  model fits in 9.1 GB.
- Temperature 0 repeats itself. The model is stateless; `chat` works because
  you send the history.
- A real model **will** supply an argument it does not have. Do not ask the
  model for values your own code can look up.
- A SYSTEM instruction changes behaviour without any training.

## Exercises

1. Add a system prompt to `ask` that says "Always look up a product's price
   with check_inventory before calculating a discount." Does the laptop answer
   become 49,500?
2. Remove `base_price` from the discount tool's schema and fill it yourself
   from `check_inventory`. Is there anything left for the model to invent?
3. If you have a larger model with the `tools` capability, run Part E on it.
   Does size change the answer?
4. Write a real `Modelfile` with the sentiment SYSTEM line, run
   `ollama create sentiment -f Modelfile`, and call the new model with no
   `system=` argument. Is it still 6 of 6?